# IMPORT MODULES

In [27]:
import pandas as pd
import numpy as np
import random
import itertools
import json
import pprint
import datetime
from datetime import date, time, timedelta, datetime
import requests
import os
import pandas_market_calendars as mcal
import base64
from config import appKey, appSecret
from datetime import datetime, timedelta, date
import traceback
import glob

# STRATEGY INPUTS AND VARIABLES

In [28]:
## DEFINING ALL VARIABLES FO## DEFINING ALL VARIABLES FOR BACKTESTING

######################################## STRATEGY NOTE TO REMIND ON OUTPUT ########################################
strategy_note = '50M+Float'
run = strategy_note

########################################  IMPORT LIST OF ALL TICKERS FOR BACKTEST/STRATEGY   #################################
# ticker_list = pd.read_excel('./prod_files/by_float/group_2_float_11M_to_26M.xlsx')
ticker_list = pd.concat(map(pd.read_excel, glob.glob("./prod_files/by_float/*.xlsx")))
ticker_list = ticker_list[ticker_list['Float'] > 50000000]

####################################### SETTING CANDLE TIME FRAME FOR STRATEGY ########################################

# Define variables for stock time frame for strategy
period_type = 'day'  # Current value
period = 10          # Current value
frequency_type = 'minute'  # Current value
frequency = 30      # Current value
need_extended_hours_data = 'true'  # Current value
need_previous_close = 'true'  # Current value

######################################## TESTING COMBINATION OF INPUTS FOR STRATEGY  ########################################

TOTAL_CASH = 10000 #FIXED
BET_SIZE = [.1] #FIXED
STOP = [.1] #FIXED #Removed .05
TARGET = [.05] #FIXED

VOL_SPIKE_THRESHOLD = [5] #Abnormally high volume that stands out on a chart
PRICE_SPIKE_THRESHOLD = [.05] #Must move the price x%
TIME_SIG_THRESHOLD = [time(hour=12,minute=30,second=0)]
BUY_TIME_THRESHOLD = [time(hour=10,minute=30,second=0)]
SELL_TIME_THRESHOLD = [time(hour = 15,minute = 30,second =0)]

####################################### CREATE VARIABLES FOR INPUT STRATEGY TO TEST ##########################

bet_size_index = 0
stop_index = 1
target_index = 2
vol_spike_thresh_index = 3
price_spike_thresh_index = 4
time_sig_thresh_index = 5
buy_time_threshold_index = 6
sell_time_threshold_index = 7

variables = [BET_SIZE,STOP,TARGET,VOL_SPIKE_THRESHOLD,PRICE_SPIKE_THRESHOLD,TIME_SIG_THRESHOLD,BUY_TIME_THRESHOLD, SELL_TIME_THRESHOLD]
combinations = list(itertools.product(*variables))      

######################################## STRATEGY OUTPUT TABLE ########################################

#FINAL OUTPUT TO EVALUATE DIFFERENT STRATEGY COMBINATIONS

inputs = pd.DataFrame(columns=['Account Size',
                               'Bet_Size',
                               'Stop',
                               'Target',
                               'Vol Spike Thresh',
                               'Price Spike Thresh',
                               'Signal Time Threshold',
                               'Buy Time Threshold',
                               'Sell_Time',
                               'Signals',
                               'Buys',
                               'Win %',
                               'Average Win',
                               'Strategy Note'])

########################################  TIME FRAME  ########################################

# Set date parameters and calculations regarding dates such as 10 day rolling average volume

en = datetime.now()
st = en - timedelta(days=365)

days = (en-st).days

start_time = str(int(st.timestamp())*1000)
end_time = str(int(en.timestamp())*1000)

today = date.today()

time_interval = 30
time_interval_string = str(time_interval)+'m'
rolling_window_days = 10
tickers_per_day = 60/time_interval*6.5
rolling_lookback = rolling_window_days*tickers_per_day

# Find Next Business Day
# Import the New York Stock Exchange Calendar
nyse = mcal.get_calendar('NYSE')
open_close_schedule = pd.DataFrame(nyse.schedule(start_date=st, end_date=en))
open_close_schedule.index.names = ['Date']
open_close_schedule.reset_index(inplace=True)
open_close_schedule['Date'] = open_close_schedule['Date'].dt.date
open_close_schedule['market_open'] = open_close_schedule['market_open'] - timedelta(hours=4)
open_close_schedule['market_close'] = open_close_schedule['market_close'] - timedelta(hours=4, minutes=time_interval)
open_close_schedule['market_open'] = open_close_schedule['market_open'].dt.time
open_close_schedule['market_close'] = open_close_schedule['market_close'].dt.time

nyse_days = nyse.valid_days(start_date=st, end_date=en)
valid_trading_days = pd.Series(nyse_days).dt.date

schedule = pd.DataFrame(nyse.schedule(start_date=st, end_date=en))


In [29]:
######################################### SCHWAB API FUNCTIONS ########################################################
def get_schwab_access_token():
    authUrl = f'https://api.schwabapi.com/v1/oauth/authorize?client_id={appKey}&redirect_uri=https://127.0.0.1'
    print(f"Click to authenticate: {authUrl}")
    returnedLink = input("Paste the redirect URL here:")
    code = f"{returnedLink[returnedLink.index('code=')+5:returnedLink.index('%40')]}@"

    headers = {
        'Authorization': f'Basic {base64.b64encode(bytes(f"{appKey}:{appSecret}", "utf-8")).decode("utf-8")}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    data = {
        'grant_type': 'authorization_code',
        'code': code,
        'redirect_uri': 'https://127.0.0.1'
    }

    response = requests.post('https://api.schwabapi.com/v1/oauth/token', headers=headers, data=data)
    token_data = response.json()
    return token_data['access_token']


def auto_authenticate():
    token_file = 'schwab_token.json'
    
    # Check if token file exists and is not expired
    if os.path.exists(token_file):
        with open(token_file, 'r') as f:
            token_data = json.load(f)
        
        expires_at = datetime.fromisoformat(token_data['expires_at'])
        if expires_at > datetime.now():
            return token_data['access_token']
    
    # If no valid token, authenticate
    headers = {
        'Authorization': f'Basic {base64.b64encode(bytes(f"{appKey}:{appSecret}", "utf-8")).decode("utf-8")}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    data = {
        'grant_type': 'client_credentials',
        'scope': 'openid'
    }

    response = requests.post('https://api.schwabapi.com/v1/oauth/token', headers=headers, data=data)
    if response.status_code != 200:
        raise Exception(f"Authentication failed: {response.text}")

    token_data = response.json()
    # Convert expires_in to an integer before using it
    expires_in = int(token_data.get('expires_in', 3600))  # Default to 1 hour if not present
    token_data['expires_at'] = (datetime.now() + timedelta(seconds=expires_in)).isoformat()

    # Save token data
    with open(token_file, 'w') as f:
        json.dump(token_data, f)

    return token_data['access_token']

def get_stock_price_history(symbol, access_token, period_type, period, frequency_type, frequency, need_extended_hours_data, need_previous_close):
    url = 'https://api.schwabapi.com/marketdata/v1/pricehistory'
    params = {
        'symbol': symbol,
        'periodType': period_type,
        'period': period,
        'frequencyType': frequency_type,
        'frequency': frequency,
        'needExtendedHoursData': need_extended_hours_data,
        'needPreviousClose': need_previous_close
    }
    headers = {'Authorization': f'Bearer {access_token}'}
    
    response = requests.get(url, params=params, headers=headers)
    return response.json()

    #DOCUMENTATION FOR API CALL
    # If the periodType is
    # • day - valid values are 1, 2, 3, 4, 5, 10
    # • month - valid values are 1, 2, 3, 6
    # • year - valid values are 1, 2, 3, 5, 10, 15, 20
    # • ytd - valid values are 1

    # If the period is not specified and the periodType is
    # • day - default period is 10.
    # • month - default period is 1.
    # • year - default period is 1.
    # • ytd - default period is 1.

    # period
    # frequencyType
    # string
    # (query)
    # The time frequencyType

    # If the periodType is
    # • day - valid value is minute
    # • month - valid values are daily, weekly
    # • year - valid values are daily, weekly, monthly
    # • ytd - valid values are daily, weekly

    # If frequencyType is not specified, default value depends on the periodType
    # • day - defaulted to minute.
    # • month - defaulted to weekly.
    # • year - defaulted to monthly.
    # • ytd - defaulted to weekly.

    # Available values : minute, daily, weekly, monthly


    # --
    # frequency
    # integer($int32)
    # (query)
    # The time frequency duration

    # If the frequencyType is
    # • minute - valid values are 1, 5, 10, 15, 30
    # • daily - valid value is 1
    # • weekly - valid value is 1
    # • monthly - valid value is 1

    # If frequency is not specified, default value is 1

    # frequency
    # startDate
    # integer($int64)
    # (query)
    # The start date, Time in milliseconds since the UNIX epoch eg 1451624400000
    # If not specified startDate will be (endDate - period) excluding weekends and holidays.

    # startDate
    # endDate
    # integer($int64)
    # (query)
    # The end date, Time in milliseconds since the UNIX epoch eg 1451624400000
    # If not specified, the endDate will default to the market close of previous business day.

    # Can use this API Wrapper or take as inspiration: https://github.com/tylerebowers/Schwab-API-Python/blob/main/docs/stream.md
####Create Function for input stats to track to easily just change the list
# # all STRATEGY INPUTS AND VARIABLES should be put into different config files



In [30]:
# Download data into memory
if 'Ticker' in ticker_list.columns:
    # Create a new DataFrame with only the 'Tickers' column
    tickers_df = ticker_list[['Ticker']].dropna()
    # Convert to a list of unique tickers
    tickers = tickers_df['Ticker'].unique().tolist()  # Extract unique tickers
    print(tickers)  # Display the list of tickers
    print('SUCESS IF YOU SEE TICKERS!!! ^^^^^^^')
else:
    print("Column 'Ticker' not found in the DataFrame.")

['BL', 'ALKT', 'STEW', 'CRL', 'ZBRA', 'PASG', 'AX', 'KFY', 'JKS', 'HCC', 'MOD', 'LMND', 'AIZ', 'TRMD', 'SNA', 'THS', 'AREC', 'CENX', 'VSTO', 'GDOT', 'WFG', 'DRTS', 'WDI', 'TY', 'SGML', 'CBU', 'CLYM', 'EBS', 'MHK', 'HG', 'VBTX', 'PLTK', 'MGTX', 'TFSL', 'GRNT', 'OPI', 'RGEN', 'VTEX', 'ACP', 'MHD', 'ZG', 'PII', 'ZVRA', 'EOS', 'REPL', 'KLIC', 'ARW', 'HYPR', 'MLTX', 'SOL', 'CENTA', 'QNST', 'FULC', 'PCN', 'SANM', 'PRQR', 'SUPN', 'IQI', 'RGLS', 'GKOS', 'DXLG', 'IDA', 'NNOX', 'SLRC', 'BTSG', 'MIDD', 'HLI', 'MATV', 'PRIM', 'PAVS', 'EXFY', 'CRUS', 'MUJ', 'CION', 'VUZI', 'CNTB', 'BAP', 'FIVE', 'WHR', 'WIX', 'EPAC', 'HUBB', 'BHAT', 'CRTO', 'ARBK', 'PLRX', 'CHW', 'PRTS', 'PHR', 'CBT', 'AHT', 'BHK', 'EMBC', 'ELF', 'LIVN', 'NDSN', 'LECO', 'REYN', 'VGM', 'GRWG', 'PAM', 'BUSE', 'FUL', 'ACB', 'EOLS', 'ENV', 'RYTM', 'CTOS', 'KTB', 'BZUN', 'MMU', 'IVR', 'RS', 'IBOC', 'VTYX', 'TDG', 'PXLW', 'GPCR', 'PARR', 'NMCO', 'RNST', 'NMRA', 'ONL', 'LUNR', 'PTLO', 'EPAM', 'ABVX', 'ENSG', 'AGIO', 'ENOV', 'VNDA', 'RA', 

In [31]:
################################## TESTING/AUTHENTICATING CONNECTION TO SCHWAB API ##################################
# Get access token (figure out how often to do this optimally)
access_token = auto_authenticate()

# Get AAPL price history for the last year using the defined variables
aapl_price_history = get_stock_price_history('AAPL', access_token, period_type, period, frequency_type, frequency, need_extended_hours_data, need_previous_close)

# Convert to DataFrame
aapl_df = pd.DataFrame(aapl_price_history['candles'])

# Rename columns to match the required format
aapl_df.rename(columns={
    'datetime': 'Datetime',
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
}, inplace=True)

# Convert Datetime to EST
aapl_df['Datetime'] = pd.to_datetime(aapl_df['Datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)

# Save the DataFrame as a CSV file
# aapl_df.to_csv('./file_uploads/tests/aapl_price_history.csv', index=False)

# Print the result

print(aapl_price_history)
print('SUCCESS IF I SEE APPLE STOCK DATA!!!')
print('AAPL price history saved to ./file_uploads/tests/aapl_price_history.csv')

{'candles': [{'open': 233.53, 'high': 233.6, 'low': 233.17, 'close': 233.25, 'volume': 29106, 'datetime': 1729162800000}, {'open': 233.33, 'high': 233.79, 'low': 233.27, 'close': 233.4985, 'volume': 29188, 'datetime': 1729164600000}, {'open': 233.55, 'high': 233.81, 'low': 231.201, 'close': 233.59, 'volume': 49020, 'datetime': 1729166400000}, {'open': 233.59, 'high': 233.76, 'low': 233.3, 'close': 233.76, 'volume': 30986, 'datetime': 1729168200000}, {'open': 233.792, 'high': 234.19, 'low': 232.72, 'close': 233.4395, 'volume': 191292, 'datetime': 1729170000000}, {'open': 233.43, 'high': 233.85, 'low': 230.52, 'close': 231.72, 'volume': 3725165, 'datetime': 1729171800000}, {'open': 231.76, 'high': 232.58, 'low': 231.29, 'close': 232.4139, 'volume': 1684855, 'datetime': 1729173600000}, {'open': 232.41, 'high': 232.4899, 'low': 231.16, 'close': 231.3599, 'volume': 1272939, 'datetime': 1729175400000}, {'open': 231.36, 'high': 232.321, 'low': 231.23, 'close': 232.26, 'volume': 843060, 'datet

# Get All Stock Data, Calculate Relevant Stats, and Store in DF's

In [32]:

######################### CREATE INITIAL DATAFRAME OF STOCK PRICE HISTORY AND PERFORM PRECALCULATIONS #########################

start_clock = datetime.now()  # calculate run time
go = 1
stockies = {} #Create dataframes of stock data for iteration

for ticker in tickers:
    # Get stock price history from Schwab API
    stock_data = get_stock_price_history(ticker, access_token, period_type, period, frequency_type, frequency, need_extended_hours_data, need_previous_close)
    
    if not stock_data or 'candles' not in stock_data:
        print(f"No data available for {ticker}")
        continue
    
    # Convert to DataFrame
    stahks = pd.DataFrame(stock_data['candles'])
    
    # Ensure all required fields are present
    required_fields = ['datetime', 'open', 'high', 'low', 'close', 'volume']
    if not all(field in stahks.columns for field in required_fields):
        print(f"Missing required fields for {ticker}. Available columns: {stahks.columns}")
        continue
    
    # Rename columns to match the required format
    stahks.rename(columns={
        'datetime': 'Datetime',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    }, inplace=True)
    
    # Convert Datetime to EST
    stahks['Datetime'] = pd.to_datetime(stahks['Datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)
    
    # Print the DataFrame to inspect its structure
    # print(f"Data for {ticker}:")
    # print(stahks.head())  # Display the first few rows of the DataFrame
    # print("Columns in DataFrame:", stahks.columns)  # Print the column names
    
    #Skip stock if there is insufficient amount of data (data for each period of normal trading hours)
    end_check = stahks['Datetime'].max()
    start_check = stahks['Datetime'].min()
    daydiff = end_check.weekday() - start_check.weekday()
    days = ((end_check-start_check).days - daydiff) / 7 * 5 + min(daydiff,5) - (max(end_check.weekday() - 4, 0) % 5)
    
    print(ticker, len(stahks.index))    
    # Removed the erroneous line that referenced 'datetime' instead of 'Datetime'
    # stahks['datetime'] = pd.to_datetime(stahks['datetime']/1000, unit = 's')-timedelta(hours =4)
    # stahks.columns = ['Open','High','Low','Close','Volume','Datetime']
    
    #Rearrange Columns and Merge with Open/Close Schedule
    stahks['Ticker'] = ticker
    stahks['Date'] = stahks['Datetime'].dt.date
    stahks = stahks.merge(open_close_schedule,how = 'left', on = 'Date')

    # Only calculate and average volume if there is sufficient data
    rolling_lookback_int = int(rolling_lookback)
    stahks['10_Day_Avg_Vol'] = stahks.Volume.rolling(rolling_lookback_int, min_periods=rolling_lookback_int).mean()
    stahks['10_Day_Avg_Vol'] = stahks['10_Day_Avg_Vol'].fillna(float('inf'))
    stahks['Time'] = stahks['Datetime'].dt.time
    
    #Remove any accidental duplicates (FIGURE OUT WHY????)
    stahks.drop_duplicates(['Ticker','Date','Time'],inplace = True,ignore_index=True)
    
    if len(stahks.index) < (days * tickers_per_day):
        continue
        
    #Create column with the open bar's low price (For % gap up calculation with spike later in day)
    cond = (stahks['Time'] == stahks['market_open'])
    stahks['Day_Open_Low'] = stahks[cond].groupby('Date',as_index=True)['Low'].transform('min').ffill()
    stahks['After Hours'] = (stahks['Time'] > stahks['market_close']) | (stahks['Time'] < stahks['market_open'])
    cond_2 = (stahks['After Hours'] == True)
    stahks['Pre-Market High'] = stahks[cond_2].groupby('Date',as_index=True)['High'].transform('max')
    
    stahks = stahks.ffill(axis=0)
    stahks = stahks.bfill(axis=0)

    
    stahks['VWAP_Row'] = stahks['Volume']*((stahks['High']+stahks['Low']+stahks['Close'])/3)
    stahks['Cum_VWAP'] = stahks.groupby('Date')['VWAP_Row'].transform('cumsum')
    stahks['Cum_Volume'] = stahks.groupby('Date')['Volume'].transform('cumsum')
    stahks['VWAP'] = stahks['Cum_VWAP']/stahks['Cum_Volume']
    stahks['VWAP_STD_1'] = stahks['VWAP'] - stahks.groupby('Date')['VWAP'].transform('std')
    stahks['Color_Bar'] = np.where(stahks['Open']<=stahks['Close'], 'Green', 'Red')
    #vol_window = 1
    stahks['Day_Close'] = (stahks['Time'] == stahks['market_close'])

    
    stockies[ticker] = pd.DataFrame(stahks, columns=stahks.keys())
    print(f"{ticker} processed successfully.")
    go += 1

    #Calculate pre-market volume for day 
    #Calculate pre-market change for the day 
    #stockies['Ticker_Return'] = (stahks['Close']/stahks['Close'].shift(vol_window))-1
    #stockies['Rolling_Vol'] = stahks['Ticker_Return'].std(ddof=130)
    
    ##PRINT OUT THE DATAFRAME TO A CSV FILE
    # stahks.to_csv(f"results/test/{ticker}_stahks_data.csv", index=False, header=True)

#Stockies is the final dataframe that will be used for the screen




BL 146
BL processed successfully.
ALKT 167
ALKT processed successfully.
STEW 138
STEW processed successfully.
CRL 161
CRL processed successfully.
ZBRA 149
ZBRA processed successfully.
PASG 170
PASG processed successfully.
AX 150
AX processed successfully.
KFY 143
KFY processed successfully.
JKS 292
JKS processed successfully.
HCC 160
HCC processed successfully.
MOD 159
MOD processed successfully.
LMND 210
LMND processed successfully.
AIZ 149
AIZ processed successfully.
TRMD 254
TRMD processed successfully.
SNA 151
SNA processed successfully.
THS 142
THS processed successfully.
AREC 164
AREC processed successfully.
CENX 161
CENX processed successfully.
VSTO 150
VSTO processed successfully.
GDOT 147
GDOT processed successfully.
WFG 140
WFG processed successfully.
DRTS 76
WDI 149
WDI processed successfully.
TY 127
TY processed successfully.
SGML 173
SGML processed successfully.
CBU 142
CBU processed successfully.
CLYM 144
CLYM processed successfully.
EBS 209
EBS processed successfully.
MH

# DATA PROCESSING AND SIGNALING

In [35]:
RESULT_INDEXER = 0
COMBO_INDEXER = 0
stocks_to_trade = pd.DataFrame(columns=['Strategy','Date','Ticker','Target Entry','Volume Spike','Price Spike','Previous Day Close','Signal Time', 'Shares', 'Stop Price', 'Sell Price', 'Backup Sell Time'])

for strategy in combinations:
    print(strategy,(datetime.now() - start_clock))
    tickers = list(stockies.keys())
    for ticker in tickers:

###########################################  CALCULATE SIGNALS FOR BACKTEST  ###########################################

        #Checks for Highest Daily Volume
        high_vol_sig = np.where(stockies[ticker]['Volume'] == stockies[ticker].groupby('Date')['Volume'].transform('max'),'True','False')
        stockies[ticker]['high_vol_sig'] = high_vol_sig
        #Checks for price spike 10x greater than 
        price_sig = np.where((stockies[ticker]['High']-stockies[ticker]['Day_Open_Low'])/stockies[ticker]['Day_Open_Low'] >= strategy[price_spike_thresh_index],'True','False')
        stockies[ticker]['price_sig'] = price_sig
        #Checks that the spike was before 12 pm (tied to highest daily volume)
        before_time = np.where(stockies[ticker]['Time'] <= strategy[time_sig_thresh_index], 'True','False')
        stockies[ticker]['before_time'] = before_time 
        #Checks that spike was a green bar
        green_bar = np.where(stockies[ticker]['Color_Bar'] == 'Green', 'True','False')
        stockies[ticker]['green_bar'] = green_bar
        #Checks for volume spike 10x greater than 10day average
        vol_spike_sig = np.where(stockies[ticker]['Volume'] > strategy[vol_spike_thresh_index] * stockies[ticker]['10_Day_Avg_Vol'],'True','False')
        stockies[ticker]['vol_spike_sig'] = vol_spike_sig
        #Checks that the spike high was the highest price of the day
        high_price_sig = np.where(stockies[ticker]['High'] >= stockies[ticker].groupby('Date')['Close'].transform('max'),'True','False')
        stockies[ticker]['high_price_sig'] = high_price_sig
        #Checks to see if it time is equal to the sell_time threshold set by user
        stockies[ticker]['sell_time'] = np.where(stockies[ticker]['Time'] == strategy[sell_time_threshold],'True','False')

        #Checks all conditions
        stockies[ticker]['Grab_Price_Signal'] = np.where((high_vol_sig == 'True') & (vol_spike_sig == 'True') & (price_sig == 'True') & (high_price_sig == 'True') & (green_bar == 'True') & (before_time == 'True'),'True','False')

    ################################ INPUT BUY AND SELL SIGNALS  ##################################################

        #Find a way to turn on and off signals and rules to be 'True' 'False' 'Ignore' - yet still works with backtest framework

        #Checks that the close was lower than the VWAP (maybe in backtest)
        stockies[ticker]['Close_Condition'] = 'False'

        #Temp Variables for backtest (row by row iteration)
        #Create temporary signal day variable that signaled whether or not the price_to_buy_signal was triggered during that day
        TEMP_SIGNAL_DAY = stockies[ticker]['Date'][0] - timedelta(days=10)

        #Initially set not to trigger and gets set on price_buy_signal
        TARGET_ENTRY_PRICE = 0 
        TARGET_ENTRY_PRICE_2 = 0
        TEMP_SIGNAL_TIME = time(hour = 19, minute = 30, second = 0)
        
        VOLUME_SPIKE = 0
        PRICE_SPIKE = 0

        #Iterate over rows to see which rows meet the close condition and the all clear to buy signal (pending final signal: price cross)
        #Unique to this strategy's backtest. Could be a part of inserting variables and signals before BACKTEST SECTION
        
        for index, row in stockies[ticker].iterrows(): 
            if (row['Grab_Price_Signal'] == 'True') & (row['Date'] == en.date()):
                TEMP_SIGNAL_DAY = row['Date']
                TARGET_ENTRY_PRICE = row['VWAP']
                TEMP_SIGNAL_TIME = row['Time']
                VOLUME_SPIKE = row['Volume']/row['10_Day_Avg_Vol']
                PRICE_SPIKE = (row['High']-row['Day_Open_Low'])/row['Day_Open_Low']
                
            if (row['Close']<=TARGET_ENTRY_PRICE) & (row['Day_Close'] == True) & (row['Date'] == TEMP_SIGNAL_DAY):
                print('Got in thur')
                stocks_to_trade.at[RESULT_INDEXER,'Previous Day Close'] = row['Close']
                stocks_to_trade.at[RESULT_INDEXER,'Strategy'] = COMBO_INDEXER
                stocks_to_trade.at[RESULT_INDEXER,'Ticker'] = row['Ticker']
                stocks_to_trade.at[RESULT_INDEXER,'Shares'] = ACCOUNT_SIZE*ALLOCATION/TARGET_ENTRY_PRICE
                stocks_to_trade.at[RESULT_INDEXER,'Target Entry'] = TARGET_ENTRY_PRICE
                stocks_to_trade.at[RESULT_INDEXER,'Date'] = row['Date']
                stocks_to_trade.at[RESULT_INDEXER,'Signal Time'] = TEMP_SIGNAL_TIME
                stocks_to_trade.at[RESULT_INDEXER,'Volume Spike'] = VOLUME_SPIKE
                stocks_to_trade.at[RESULT_INDEXER,'Price Spike'] = PRICE_SPIKE
                stocks_to_trade.at[RESULT_INDEXER,'Time Threshold'] = strategy[time_sig_thresh_index]
                stocks_to_trade.at[RESULT_INDEXER,'Sell_Time'] = strategy[sell_time_threshold_index]
                
                # Calculate Stop Price and Sell Price
                stop_price = TARGET_ENTRY_PRICE + (TARGET_ENTRY_PRICE * strategy[stop_index])
                sell_price = TARGET_ENTRY_PRICE - (TARGET_ENTRY_PRICE * strategy[target_index])
                stocks_to_trade.at[RESULT_INDEXER,'Stop Price'] = stop_price
                stocks_to_trade.at[RESULT_INDEXER,'Sell Price'] = sell_price

                RESULT_INDEXER +=1
                
    COMBO_INDEXER +=1
    
stocks_to_trade = pd.merge(stocks_to_trade,ticker_list[['Ticker','Market Cap','Sector','Float']],on = 'Ticker', how = 'left')  
stocks_to_trade['Market Capitalization'] = (stocks_to_trade['Market Cap'].astype(float)/1000000).astype(str) + 'M'
stocks_to_trade['Shares Float'] = (stocks_to_trade['Float'].astype(float)/1000000).astype(str) + 'M'
stocks_to_trade.sort_values(by = 'Market Capitalization',ascending = True,inplace = True)
output_file_path = './daily_signals/' + str(date.today() + timedelta(days=1)) + '.xlsx'
stocks_to_trade.to_excel(output_file_path, index=True, header=True)

print(datetime.now() - start_clock)

(0.1, 0.1, 0.05, 5, 0.05, datetime.time(12, 30), datetime.time(10, 30), datetime.time(15, 30)) 0:17:56.667185
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
0:18:06.243231
